# 01 — Etapa de detecção e recorte

**Pergunta:** dá para melhorar o *recorte* (o YOLO que localiza os objetos) **sem retreinar**, só mudando a inferência? E qual recorte (margem/janela) ajuda mais o classificador?

Sem GPU, o retreino do YOLO ficou como kit pronto em `yolo_gpu/` (variantes G1–G6). Aqui testamos o **Modelo A** já treinado (YOLOv8n-P2, 640 px) com 5 formas de inferência:

| Config | O que muda |
|---|---|
| D1_640 | Referência (como no notebook) |
| D2_960 / D3_1280 | Imagem ampliada na inferência |
| D4_640_tta | *Test-time augmentation* (flip + escalas) |
| D5_tiles2x2_640 | Imagem cortada em 4 blocos com sobreposição + imagem inteira |

### Protocolo comum
- **Dados:** SIMD (Marinha Portuguesa, 1170 imagens side-scan, 2010–2021). **MILCO** = objeto com aparência de mina; **NOMBO** = objeto que não é mina.
- **Split:** idêntico ao notebook `sonar_deteccao_comparacao_modelos_V3_AA.ipynb` (arquivos ordenados por nome, `random.Random(42).shuffle`, 70/15/15) → 819 / 175 / 176 imagens. No teste há **109 objetos** (75 MILCO, 34 NOMBO).
- **Seleção sempre na validação** (checkpoint, conf do detector, limiar de aceite, finalista). O teste é usado uma vez, para reportar.
- **Hardware:** máquina local com 2 núcleos de CPU e 4 GB de RAM, **sem GPU**; treinos fatiados e retomáveis (`crop_harness.py` + `fila_recortes.json`).
- **Pesos antigos** (YOLO Modelo A e fusão GhostNetV2 + MobileNetV2) foram apenas lidos, nunca alterados.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3); pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 90)
sys.path.insert(0, str(Path.cwd()))
import analise_recortes as A

## Varredura de detecção (conf 0,25; AP50 com IoU 0,5)

In [2]:
A.resumo_deteccao(conf=0.25)

,tag,iou_nms,AP50 test,AP50 val,FP_por_img test,FP_por_img val,recall test,recall val
9,D4_640_tta,0.5,0.744,0.722,0.193,0.109,0.734,0.653
10,D4_640_tta,0.6,0.741,0.714,0.210,0.126,0.734,0.653
11,D4_640_tta,0.7,0.725,0.698,0.307,0.160,0.734,0.653
1,D1_640,0.6,0.700,0.668,0.153,0.069,0.670,0.587
0,D1_640,0.5,0.701,0.667,0.153,0.069,0.670,0.587
2,D1_640,0.7,0.699,0.658,0.170,0.074,0.670,0.587
12,D5_tiles2x2_640,0.5,0.621,0.640,0.449,0.246,0.716,0.707
13,D5_tiles2x2_640,0.6,0.619,0.631,0.494,0.257,0.725,0.707
14,D5_tiles2x2_640,0.7,0.606,0.603,0.614,0.331,0.725,0.707
3,D2_960,0.5,0.525,0.532,0.199,0.154,0.532,0.493


**Leitura:**
- **TTA (D4) é a melhor opção:** AP50 de validação 0,722 (contra 0,668 da referência) e recall de teste de 0,67 → 0,73, com poucos falsos positivos extras.
- **Ampliar a imagem piora muito** (D2, D3): o modelo foi treinado em 640 px e não generaliza para objetos "maiores".
- **Blocos (D5)** têm o maior recall de validação, mas dobram os falsos positivos. Só valem a pena com um classificador que rejeite fundo (testado no notebook 03).
- Escolha (na validação): **D4_640_tta com NMS 0,5**.

## Forma do recorte para o classificador (grupo B3, ConvNeXt-Pico 224)

In [3]:
t = A.tabela_classificadores(["B1", "B3"])
t[t.job.isin(["b1_convnextpico_224", "b3_margem030", "b3_margem050", "b3_quadrado2x"])]

,job,grupo,descrição,val AUC,teste acc,teste F1 macro,teste AUC,teste acc (TTA),épocas,"treino (min, CPU)",ms/recorte (CPU)
1,b1_convnextpico_224,B1,"ConvNeXt-Pico 224 px, fine-tuning completo, AdamW+cosine",0.946,0.899,0.879,0.932,0.908,22,16.0,24.3
4,b3_margem030,B3,Recorte com margem 30%,0.938,0.789,0.778,0.940,0.761,19,13.2,24.6
5,b3_margem050,B3,Recorte com margem 50%,0.927,0.917,0.900,0.948,0.936,30,20.3,27.8
6,b3_quadrado2x,B3,Janela quadrada (2× o maior lado da caixa),0.898,0.881,0.858,0.933,0.881,13,8.7,22.9


**Leitura:** a margem de 15% (a do notebook antigo) tem a melhor AUC de validação. Margens maiores dão resultados de teste instáveis: 30% cai para 0,79 de acurácia e 50% sobe para 0,92 com uma semente só. Ou seja, com 109 objetos no teste, essas diferenças estão dentro do ruído. Ficamos com **m0.15**.

In [4]:
# (Opcional) Re-treinar os experimentos deste notebook. Os resultados abaixo já vêm de resultados/runs/.
# Em máquina com GPU, troque RODAR_TREINO para True: o mesmo harness treina sem limite de tempo.
RODAR_TREINO = False
GRUPOS = ["B3"]
import crop_harness as CH
fila = json.load(open("fila_recortes.json"))
jobs = [j for j in fila if j.get("group") in GRUPOS]
for j in jobs:
    print(f"{j['id']:32s} {j.get('desc','')}")
if RODAR_TREINO:
    import math
    for j in jobs:
        CH.train_job(j, deadline=math.inf)

b3_margem030                     Recorte com margem 30%
b3_margem050                     Recorte com margem 50%
b3_quadrado2x                    Janela quadrada (2× o maior lado da caixa)
